# Мини-запуск MPP: небольшой трансформер + Trainer

Загружаем срез данных, собираем маленькую MaskedPlayerModel и гоняем несколько шагов через HuggingFace Trainer.

In [1]:
import sys
from pathlib import Path

ROOT = Path(".").resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")  # Apple Silicon
else:
    device = torch.device("cpu")
print("Device:", device)
# AMD GPU: нужен PyTorch с ROCm — https://pytorch.org/get-started/locally/ (выбрать ROCm)

Device: mps


## 1. Данные и словари

In [2]:
from data.preprocessing import preprocess_raw_csv, build_vocab_mappings

raw_path = ROOT / "dataset" / "data_with_dates.csv"
sample_path = ROOT / "notebooks" / "train_sample_raw.csv"
output_dir = str(ROOT / "notebooks" / "train_sample_processed")

df_raw = pd.read_csv(raw_path)
df_raw.to_csv(sample_path, index=False)
df = preprocess_raw_csv(str(sample_path), output_dir)
vocab = build_vocab_mappings(df, output_dir)

print("Матчей (уникальных match_id):", df["match_id"].nunique())
print("players_vocab_size (pad_idx+1):", vocab["player_pad_token_id"] + 1)
print("Число классов для MPP (реальные игроки):", vocab["player_pad_token_id"] - 1)

Матчей (уникальных match_id): 2923
players_vocab_size (pad_idx+1): 6395
Число классов для MPP (реальные игроки): 6393


In [3]:
TARGET_COLUMNS = [
    "pass_total",
    "pass_cross",
    "pass_shot_assist",
    "pass_goal_assist",
    "pass_through_ball",
    "shot_total",
    "shot_statsbomb_xg",
    "shot_goal",
    "interception_won",
    "block_total",
    "clearance_total",
    "ball_recovery_total",
    "counterpress_total",
    "dribble_complete",
    "foul_won_total",
    "foul_committed_total",
    "goalkeeper_save",
    "shot_saved",
]

print("Число target stats:", len(TARGET_COLUMNS))

Число target stats: 18


In [4]:
from data.dataset import MatchDatasetNMSP
from data.collator import DataCollatorNMSP
from torch.utils.data import Subset

max_seq_length = 36

ds_full = MatchDatasetNMSP(
    df=df,
    target_columns=TARGET_COLUMNS,
    player_name2id=vocab["player_name2id"],
    team_name2id=vocab["team_name2id"],
    max_seq_length=max_seq_length,
    player_pad_token_id=vocab["player_pad_token_id"],
    team_pad_token_id=vocab["team_pad_token_id"],
    position_pad_token_id=25,
)

n = len(ds_full)
n_val = max(1, int(n * 0.05))
n_train = n - n_val

train_dataset = Subset(ds_full, range(0, n_train))
eval_dataset = Subset(ds_full, range(n_train, n))

collator = DataCollatorNMSP()

print("Train матчей:", n_train, "Eval матчей:", n_val)
print("form_stats_size:", ds_full.form_stats_size)

Train матчей: 2777 Eval матчей: 146
form_stats_size: 21


In [5]:
sample = ds_full[0]
for k, v in sample.items():
    print(k, v.shape, v.dtype)

input_ids torch.Size([36]) torch.int64
position_id torch.Size([36]) torch.int64
team_id torch.Size([36]) torch.int64
form_stats torch.Size([36, 21]) torch.float32
attention_mask torch.Size([36]) torch.int64
target_stats torch.Size([36]) torch.float32


In [6]:
batch = collator([ds_full[0], ds_full[1]])
for k, v in batch.items():
    print(k, v.shape, v.dtype)

input_ids torch.Size([2, 36]) torch.int64
position_id torch.Size([2, 36]) torch.int64
team_id torch.Size([2, 36]) torch.int64
form_stats torch.Size([2, 36, 21]) torch.float32
attention_mask torch.Size([2, 36]) torch.int64
target_stats torch.Size([2, 36]) torch.float32


In [7]:
from models.finetune import DownstreamModel

embed_size = 128

encoder_config = {
    "embed_size": embed_size,
    "num_layers": 1,
    "heads": 2,
    "forward_expansion": 4,
    "dropout": 0.05,
    "form_stats_size": ds_full.form_stats_size,
    "players_vocab_size": vocab["player_pad_token_id"],
    "teams_vocab_size": vocab["team_pad_token_id"],
    "positions_vocab_size": 25,
    "use_teams_embeddings": False,
    "position_enc_type": "learned",
}

head_config = {
    "type": "nmsp",
    "max_seq_length": 36,
    "num_target_stats": len(TARGET_COLUMNS),
    "hidden_dim": 512,
}

model = DownstreamModel(
    encoder_config=encoder_config,
    head_config=head_config,
    freeze_encoder=False,
)
model = model.to(device)

print("Параметров модели:", sum(p.numel() for p in model.parameters()))

Параметров модели: 3364004


In [8]:
batch = collator([ds_full[0], ds_full[1]])
batch = {k: v.to(device) for k, v in batch.items()}

out = model(**batch)

print(out.keys())
print("predictions shape:", out["predictions"].shape)
print("loss:", out["loss"])

dict_keys(['predictions', 'loss', 'hidden_states', 'attentions'])
predictions shape: torch.Size([2, 36])
loss: tensor(15726.1143, device='mps:0', grad_fn=<MseLossBackward0>)


In [ ]:
from training.trainer import build_training_args, build_trainer
from training.metrics import compute_metrics_nmsp

training_config = {
    "output_dir": str(ROOT / "notebooks" / "nmsp_mini_output"),
    "num_train_epochs": 1,
    "per_device_train_batch_size": 64,
    "per_device_eval_batch_size": 64,
    "learning_rate": 1e-4,
    "weight_decay": 0.0,
    "warmup_ratio": 0.0,
    "lr_scheduler_type": "linear",
    "logging_steps": 50,
    "eval_strategy": "steps",
    "eval_steps": 50,
    "save_strategy": "steps",
    "save_steps": 500,
    "report_to": "tensorboard",
    "seed": 42,
}

In [ ]:
train_args = build_training_args(training_config)

trainer = build_trainer(
    model=model,
    args=train_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics_nmsp,
    data_collator=collator,
)

print("NMSP trainer готов")

In [ ]:
trainer.train()

## 2. Датасет и коллатор

In [4]:
from data.dataset import MatchDatasetMPP
from data.collator import DataCollatorMPP
from torch.utils.data import Subset

max_seq_length = 36
ds_full = MatchDatasetMPP(
    df,
    player_name2id=vocab["player_name2id"],
    team_name2id=vocab["team_name2id"],
    max_seq_length=max_seq_length,
    player_pad_token_id=vocab["player_pad_token_id"],
    team_pad_token_id=vocab["team_pad_token_id"],
    position_pad_token_id=25,
)

n = len(ds_full)
n_val = max(1, int(n * 0.05))  # 5% eval, как в risingBALLER
n_train = n - n_val
train_dataset = Subset(ds_full, range(0, n_train))
eval_dataset = Subset(ds_full, range(n_train, n))

collator = DataCollatorMPP(
    player_mask_token_id=vocab["player_mask_token_id"],
    mask_percentage=0.25,
)

print("Train матчей:", n_train, "Eval матчей:", n_val)

Train матчей: 2777 Eval матчей: 146


## 3. Маленькая модель и TrainingArguments

In [7]:
from models.pretrain import MaskedPlayerModel
from training.trainer import build_training_args, build_trainer
from training.metrics import compute_metrics_mpp

embed_size = 128
model = MaskedPlayerModel(
    embed_size=embed_size,
    num_layers=1,
    heads=2,
    forward_expansion=4,
    dropout=0.05,
    form_stats_size=39,
    players_vocab_size=vocab["player_pad_token_id"],
    teams_vocab_size=vocab["team_pad_token_id"],
    positions_vocab_size=25,
    use_teams_embeddings=False,
    position_enc_type="learned",
)
model = model.to(device)

# Параметры как в risingBALLER (train_masked_players_prediction.ipynb)
training_config = {
    "output_dir": str(ROOT / "notebooks" / "mpp_mini_output"),
    "num_train_epochs": 2000,
    "per_device_train_batch_size": 64,
    "per_device_eval_batch_size": 64,
    "learning_rate": 1e-4,
    "weight_decay": 0.0,
    "warmup_ratio": 0.0,
    "lr_scheduler_type": "linear",
    "logging_steps": 100,
    "eval_strategy": "steps",
    "eval_steps": 100,
    "save_strategy": "steps",
    "save_steps": 42750,
    "report_to": "tensorboard",
    "seed": 42,
}

train_args = build_training_args(training_config)
trainer = build_trainer(
    model=model,
    args=train_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics_mpp,
    data_collator=collator,
)

print("Trainer готов. Параметров модели:", sum(p.numel() for p in model.parameters()))

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainer готов. Параметров модели: 1849977


## 4. Запуск обучения

In [8]:
trainer.train()
# Финальная валидация по eval_steps. Отдельно evaluate() после train() в Jupyter ломает NotebookProgressCallback.

Step,Training Loss,Validation Loss,Accuracy Top1,Accuracy Top3
100,8.647963,8.872501,0.000000,0.003425
200,8.237054,8.941559,0.000000,0.002283
300,8.003448,9.000739,0.000000,0.000000
400,7.787250,8.894636,0.005708,0.005708
500,7.575638,8.831105,0.003425,0.003425
600,7.372207,8.724623,0.002283,0.002283
700,7.174036,8.633806,0.003425,0.004566
800,7.003729,8.607018,0.002283,0.009132
900,6.862025,8.621562,0.002283,0.010274
1000,6.722913,8.595945,0.007991,0.014840


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f9116e6ebf0>>
Traceback (most recent call last):
  File "/home/mlcore/.local/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 

KeyboardInterrupt



In [ ]:
# Опционально: повторный eval после train. Сначала убираем NotebookProgressCallback (он обнуляется в on_train_end).
from transformers.utils.notebook import NotebookProgressCallback
trainer.remove_callback(NotebookProgressCallback)
trainer.evaluate()